In [12]:
using DualNumbers
using ForwardDiff
using LinearAlgebra
using Quadmath

In [13]:
toRadians = pi/180
numberOfAtoms::Int64 = 6
masses = [12.00000000, 15.99491463, 1.00782503223, 1.00782503223, 1.00782503223, 1.00782503223]
valenceCoordinates = [1.42167426, 0.95631445, 1.09061209, 1.09061209, 1.09061209, 108.10210467*toRadians, 110.28974516*toRadians, 110.28974516*toRadians, 110.28974516*toRadians, 0.0, 0.0, pi/3]

12-element Vector{Float64}:
 1.42167426
 0.95631445
 1.09061209
 1.09061209
 1.09061209
 1.8867376548270383
 1.9249191842274802
 1.9249191842274802
 1.9249191842274802
 0.0
 0.0
 1.0471975511965976

In [14]:
function defineMolecularFrameXYZ(valence)
    # Stores type to prevent type conversion problems with duals
    T = eltype(valence) 
    molecularFrameXYZ = zeros(T, numberOfAtoms, 3)
    # C at origin
    # CO defines z-axis
    molecularFrameXYZ[2, 3] = valence[1]
    # OH defines x-axis
    molecularFrameXYZ[3, 1] = valence[2]*sin(valence[6])
    molecularFrameXYZ[3, 3] = valence[1]-valence[2]*cos(valence[6])
    # CH1
    molecularFrameXYZ[4, 1] = valence[3]*sin(valence[7])*cos(valence[12] - sqrt(2)*valence[11]/3)
    molecularFrameXYZ[4, 2] = valence[3]*sin(valence[7])*sin(valence[12] - sqrt(2)*valence[11]/3)
    molecularFrameXYZ[4, 3] = valence[3]*cos(valence[7])
    # CH2
    molecularFrameXYZ[5, 1] = valence[4]*sin(valence[8])*cos(4*pi/3 + valence[12] + valence[10]/sqrt(6) + sqrt(2)*valence[11]/6)
    molecularFrameXYZ[5, 2] = valence[4]*sin(valence[8])*sin(4*pi/3 + valence[12] + valence[10]/sqrt(6) + sqrt(2)*valence[11]/6)
    molecularFrameXYZ[5, 3] = valence[4]*cos(valence[8])
    # CH3
    molecularFrameXYZ[6, 1] = valence[5]*sin(valence[9])*cos(2*pi/3 + valence[12] - valence[10]/sqrt(6) + sqrt(2)*valence[11]/6)
    molecularFrameXYZ[6, 2] = valence[5]*sin(valence[9])*sin(2*pi/3 + valence[12] - valence[10]/sqrt(6) + sqrt(2)*valence[11]/6)
    molecularFrameXYZ[6, 3] = valence[5]*cos(valence[9])
    
    return molecularFrameXYZ
end

function transformToCOM(molecularFrame)
    COM = masses'*molecularFrame/sum(masses)
    frameCOM = molecularFrame .- COM
    return frameCOM
end

function defineCOMframe(valence)
    molecularFrame = defineMolecularFrameXYZ(valence)
    frameCOM = transformToCOM(molecularFrame)
    return frameCOM
end

COMpositionVectorOfAtom(i) = valence -> defineCOMframe(valence)[i, :]

function computeTMatrix(valence)
    frameCOM = defineCOMframe(valence)
    T = eltype(valence)
    tMatrix = zeros(T, numberOfAtoms*3, numberOfAtoms*3)
    for i in 1:numberOfAtoms
        # Translational part
        tMatrix[i*3-2:i*3, 1:3] = Matrix(1I, 3, 3)
    
        # Rotational Part
        for j in 1:3
            unitVector = Matrix(1I, 3, 3)[j, :]
            tMatrix[i*3-2:i*3, 3 + j] = cross(unitVector, frameCOM[i, :])
        end
        # Vibrational part
        jacobianCOM = ForwardDiff.jacobian(COMpositionVectorOfAtom(i), valence)
        tMatrix[i*3-2:i*3, 7:end] = jacobianCOM
    end
    return tMatrix
end

# Redundant for these purposes - mainly useful for symbolic stuff
function computeSMatrix(valence)
    tMatrix = computeTMatrix(valence)
    sMatrix = inv(tMatrix)
    return sMatrix
end

computeSMatrixRow(i) = valence -> computeSMatrix(valence)[i, :]
computeSMatrixElement(i, j) = valence -> computeSMatrix(valence)[i, j]

# Note that the upper case GMatrix is the contravariant metric tensor! gMatrix is the covariant
function computegMatrix(valence)
    tMatrix = computeTMatrix(valence)
    T = eltype(valence)
    gMatrix = zeros(T, 3*numberOfAtoms, 3*numberOfAtoms)
    for i in 1:3*numberOfAtoms
        for j in 1:3*numberOfAtoms
            for n in 1:numberOfAtoms
                gMatrix[i, j] += dot(tMatrix[n*3-2:n*3, i], tMatrix[n*3-2:n*3, j])*masses[n]
            end          
        end
    end
    return gMatrix
end

function computeGMatrix(valence)
    gMatrix = computegMatrix(valence)
    GMatrix = inv(gMatrix)
    return GMatrix
end

computeGMatrix (generic function with 1 method)

In [16]:
@time GMatrix = computeGMatrix(valenceCoordinates)

  0.000676 seconds (14.01 k allocations: 545.891 KiB)


18×18 Matrix{Float64}:
  0.0312244    -3.64604e-35  -1.71698e-35  …   2.01333e-17   4.65811e-20
 -3.64604e-35   0.0312244     4.89351e-36     -7.12562e-18   8.1611e-19
 -1.71698e-35   4.89351e-36   0.0312244       -5.75523e-19  -5.40208e-20
  2.68391e-19  -9.61647e-19   8.96927e-20      0.0890754    -0.0719692
  2.46581e-18  -2.92935e-19  -2.62087e-19      0.154283      2.12174e-17
  2.02461e-19  -8.1611e-19    5.40208e-20  …   0.0480893    -1.3159
  1.10185e-19   8.50256e-20  -1.75391e-18      3.71552e-19   2.43931e-19
 -1.46177e-19   3.90203e-20  -7.30588e-19     -0.0283901    -4.91526e-18
  3.76879e-18   1.60941e-18  -1.22987e-18      4.94938e-17   0.0155644
  2.14973e-18  -3.74033e-18  -4.67994e-19      0.177715     -0.0155644
 -9.42522e-18   3.22628e-19   4.6217e-19   …  -0.177715     -2.56738e-17
  2.62518e-18  -3.06273e-19   2.38965e-19      0.163988      2.10249e-17
 -1.98636e-18   8.56765e-19  -2.52308e-18      2.08167e-17  -0.0676034
 -3.17775e-18  -9.97384e-19  -4.00148e-18 

In [7]:
GMatrix[18, 18]*33.7152537836138732499014581824370

55.02264342076441

In [22]:
GMatrix[18, 17]

-0.04808928757670396

In [20]:
GMatrix[4, 4]

0.07216319847977086

In [21]:
GMatrix[5, 5]

0.07216319847977086

In [23]:
GMatrix[7, 7]

0.145853204375621

In [24]:
1/sum(masses)

0.031224420604419954

In [25]:
GMatrix[1, 1]

0.031224420604419954

In [26]:
GMatrix[2, 2]

0.031224420604419954

In [27]:
GMatrix[3, 3]

0.031224420604419954

In [8]:
computeLogDetgMatrix(valence) = log(det(computegMatrix(valence)[4:end, 4:end]))
computeVibrationalGMatrix(valence) = computeGMatrix(valence)[7:end, 7:end]
function computePseudopotential(valence)
    numberOfModes::Int64 = 3*numberOfAtoms-6
    GMatrix = computeGMatrix(valence)
    hessianlogg = ForwardDiff.hessian(computeLogDetgMatrix, valence)
    gradientlogg = ForwardDiff.gradient(computeLogDetgMatrix, valence)
    jacobianGMatrix = ForwardDiff.jacobian(computeVibrationalGMatrix, valence)
    jacobianGMatrix = reshape(jacobianGMatrix, numberOfModes, numberOfModes, numberOfModes)
    jacobianGMatrix = [jacobianGMatrix[k, j, k] for k in 1:numberOfModes, j in 1:numberOfModes]
    U1 = gradientlogg'GMatrix[7:end, 7:end]*gradientlogg
    U2 = sum(jacobianGMatrix*gradientlogg) + sum(GMatrix[7:end, 7:end].*hessianlogg)
    U = (U1 + 4*U2)/32
    return U
end

computePseudopotential (generic function with 1 method)

In [19]:
@time computePseudopotential(valenceCoordinates)

  0.037659 seconds (56.41 k allocations: 44.631 MiB, 50.98% gc time)


-0.9746343471310588

In [36]:
computeGMatrix(grids[1, :])[18, 18]

60.59351102654954

In [37]:
computeGMatrix(grids[6, :])[18, 18]

60.59351102654687

In [16]:
-9.74634347131059976163210450040649138e-01*33.7152537836138732499014581824370

-32.86004435975051

In [16]:
-0.97463433030752678131409176193381*33.7152537836138732499014581824370

-32.860043792540814

In [20]:
33.7152537836138732499014581824370*1.90508494618583908786559758497174105e+00

64.23042243999794

In [99]:
valenceCoordinates[12] = 50.0*toRadians
valenceCoordinates[11] = 5.0*toRadians
valenceCoordinates[10] = 5.0*toRadians
valenceCoordinates[1] = valenceCoordinates[1]*1.2 
# valenceCoordinates[10] = 5.0*toRadians
@time computePseudopotential(valenceCoordinates)

  0.672507 seconds (22.01 k allocations: 116.878 MiB)


1.89991005665346107338706137606501703e+00

In [26]:
(33.7152537836138732499014581824370*1.90508494618590139573775017773483936)/2

32.11521122000002

In [27]:
gMatrixDet = det(computegMatrix(valenceCoordinates))
ForwardDiff.jacobian(valence -> GMatrix[7:end,7:end]*ForwardDiff.gradient(valence -> det(computeGMatrix(valence)), valence)/gMatrixDet, valenceCoordinates)

12×12 Matrix{Float128}:
  3.71577081926952743791440032757687393e-15  …  -8.39393367208879420007765749799642879e-49
  5.18280397946015298438918088356908464e-14     -1.16594185249956646524045223160560954e-47
  4.23885952197627674435492787769349327e-14     -2.54607924261543445577351188922698149e-48
  4.23885952197627676978813684110823746e-14     -4.79167366032293485267551099269132797e-48
  4.30279337232901738303567121724677857e-14     -6.30845333302202527678497430031196010e-48
 -1.43867629313614338486862384215118782e-14  …   2.90518476435380570806679031951740925e-48
 -7.23791123768376728727978174569485125e-15      1.65591590392583822634276844503672914e-48
 -7.23791123768376901050499788772978300e-15      1.96863444032474162065880394157194765e-48
 -1.15697445997803432110503364828721104e-14      2.13914848637189653366863450568373292e-48
 -1.54696039065452734588633566043749238e-15     -4.39279088133927169533682851683343671e-50
 -2.67941399391023791016484640196667114e-15  …  -1.776844142426642

In [ ]:
ForwardDiff.gradient(valence -> det(computeGMatrix(valence)), valenceCoordinates)

In [ ]:
ForwardDiff.gradient(valence -> ForwardDiff.gradient(GMatrix[7:end,7:end]*ForwardDiff.gradient(valence -> det(computeGMatrix(valence)), valence)/gMatrixDet), valenceCoordinates)

In [17]:
sMatrix = computeSMatrix(valenceCoordinates)

18×18 Matrix{Float128}:
  3.74693047253039417261521578634412272e-01  …   1.50463276905252801019998276764447447e-36
 -4.42169318976253393645805236663173886e-35     -6.68191177523048911535134116787870148e-52
 -9.02779661431516806119989660586652805e-36      3.14687527020126127975680513789426709e-02
  7.22659690853622254195893694635890494e-35     -4.32761555494170821976187211998600616e-85
 -7.03396008590603549936443804937428505e-01      0.00000000000000000000000000000000000e+00
  1.36649295735455135715608418105062005e-34  …  -8.18318255855741179361432321644802356e-85
  0.00000000000000000000000000000000000e+00      0.00000000000000000000000000000000000e+00
 -1.18390754869659320019587636232877184e-34     -0.00000000000000000000000000000000000e+00
 -4.68975507191425525283364184342959052e-01      6.09289080858463851320834252792000281e-85
 -4.68975507191424965600879523303621575e-01     -2.92434179656194232985574958274269899e-85
  9.37951014382850864005900148339447466e-01  …  -3.467677819783451

In [41]:
U1 = Float128(0.0)
for i in 1:numberOfAtoms
    for j in 1:3
        for k in 1:3
            U1 += sMatrix[3 + j, 3*i - 3 + k]*(sMatrix[3 + j, 3*i - 3 + k] - sMatrix[3 + k, 3*i - 3 + j])/(8*masses[i])
        end
    end
end

In [67]:
U2 = Float128(0.0)
for i in 1:3 # beta in TROVE paper
    sJacobian = ForwardDiff.jacobian(computeSMatrixRow(i+3), valenceCoordinates)
    for j in 1:3 # alpha in TROVE paper
        for k in 1:3 # gamma in TROVE paper
            for l in 1:numberOfAtoms*3-6
                for n in 1:numberOfAtoms
                    U2 -= leviCivitaTensor[i, j, k]*sMatrix[l, n*3 - 3 + k]*sJacobian[n*3 - 3 + j, l]/masses[n]
                end
            end
        end
    end
end
U2 = U2/4
# for i in 1:numberOfAtoms
#     for j in 1:3
#         for k in 1:numberOfAtoms-6
#             unitVector = Matrix(1I, 3, 3)[j, :]
#             cross(unitVector, sMatrix[6+k, i*3-2:i*3])*
#     end
# end

-3.46516204439887547732767308998847473e-35

In [42]:
U1

2.00569581645338610824963188998765609e-01

In [65]:
U2

2.01745895227578223807345509620946169e-02

In [94]:
Dual(Float128(0.0), Float128(1.0))

0.00000000000000000000000000000000000e+00 + 1.00000000000000000000000000000000000e+00ɛ

In [79]:
typeof(Dual(2.0, 1.0))

Dual128 (alias for Dual{Float64})

In [75]:
dualpart(tan(Dual(2, 1)))

5.774399204041917

In [167]:
ForwardDiff.derivative(x -> ForwardDiff.derivative(x -> x^2, x), Float128(2.0))

2.00000000000000000000000000000000000e+00

In [169]:
ForwardDiff.derivative(x -> ForwardDiff.derivative(x -> ForwardDiff.derivative(x -> x^3, x), x), Float128(2.0))

6.00000000000000000000000000000000000e+00